In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import sys
sys.path.append(str(Path.cwd().parent))
from mpp_project.core import load_tournament_data
from mpp_project.v5_supervised.oracle_dp import evaluate_current_match

# --- CONFIGURATION ---
MATCH_ACTUEL_ID = 1      # Le match sur lequel tu dois parier aujourd'hui
MATCH_FUTUR_ID = 15      # Le match de J2/J3 dont les cotes sont incertaines
MON_GAP_1 = 0
MON_GAP_2 = 0

data_path_csv = Path.cwd().parent / "data" / "raw" / "tournament_2026.csv"
data_path_npy = Path.cwd().parent / "data" / "processed" / "expected_V_phases_finales.npy"

# --- 1. CHARGEMENT BASELINE (Le monde tel qu'on l'estime aujourd'hui) ---
df_base, probas_base, gains_base, crowds_base = load_tournament_data(data_path_csv)
V_all = np.load(data_path_npy)

nb_poules = len(df_base[df_base['phase'] == 'Poule'])
last_idx = df_base['match_id'].max()
V_horizon = V_all[last_idx - nb_poules]

# Inférence Baseline
wr_base, ev_base = evaluate_current_match(
    MATCH_ACTUEL_ID - 1, last_idx, 
    probas_base, gains_base, crowds_base, 
    V_horizon, MON_GAP_1, MON_GAP_2
)

# --- 2. PERTURBATION (Le "Crash Test" du futur proche) ---
# On va simuler que le Match 15, qu'on pensait équilibré, devient ultra-déséquilibré 
# (ex: cotes qui passent à 1.10 / 8.00 / 20.00 et la foule suit massivement)

# On copie les matrices pour ne pas altérer la baseline
probas_pert = probas_base.copy()
crowds_pert = crowds_base.copy()

# Index du match futur (0-indexé)
idx_futur = MATCH_FUTUR_ID - 1

# Scénario Choc : Un favori écrasant apparaît en J2
probas_pert[idx_futur] = [0.85, 0.10, 0.05]
crowds_pert[idx_futur] = [0.95, 0.03, 0.02]

# Inférence Perturbée
wr_pert, _ = evaluate_current_match(
    MATCH_ACTUEL_ID - 1, last_idx, 
    probas_pert, gains_base, crowds_pert, 
    V_horizon, MON_GAP_1, MON_GAP_2
)

# --- 3. AFFICHAGE DES RÉSULTATS (L'Effet Papillon) ---
noms = ["1 (Favori)", "N (Nul)", "2 (Outsider)"]

print(f"--- ANALYSE DE SENSIBILITÉ DU MATCH {MATCH_ACTUEL_ID} ---")
print(f"Perturbation appliquée sur le Match {MATCH_FUTUR_ID} (J2/J3).")
print("-" * 50)

for i in range(3):
    diff = wr_pert[i] - wr_base[i]
    signe = "+" if diff > 0 else ""
    print(f"Choix {noms[i]:<15} : Baseline = {wr_base[i]*100:.2f}% | Nouveau = {wr_pert[i]*100:.2f}% | Écart = {signe}{diff*100:.2f}%")

action_base = np.argmax(wr_base)
action_pert = np.argmax(wr_pert)

print("-" * 50)
if action_base == action_pert:
    print(f"✅ RÉSILIENCE : La décision optimale ({noms[action_base]}) ne change pas malgré le chaos futur.")
else:
    print(f"⚠️ EFFET PAPILLON ACTIF ! La décision a basculé de '{noms[action_base]}' vers '{noms[action_pert]}'.")
    print("-> L'algorithme a détecté qu'une anomalie en J2/J3 t'oblige à changer ta prise de risque dès aujourd'hui.")